# Analyzing Historical Stock/Revenue Data and Building a Dashboard

**Companies:** Tesla (TSLA) and GameStop (GME)

This notebook contains the code for the six required lab exercises:
1. Extract Tesla stock data using `yfinance`.
2. Extract Tesla quarterly revenue using web scraping.
3. Extract GameStop stock data using `yfinance`.
4. Extract GameStop quarterly revenue using web scraping.
5. Plot Tesla stock price.
6. Plot GameStop stock price.

Run each cell in order. Internet access is required for the data-download cells. Check the resulting tables and charts before taking screenshots for Coursera.

## Setup

In [ ]:
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

print("Setup complete.")

## Question 1 — Extract Tesla stock data using yfinance

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2 — Extract Tesla revenue data using web scraping

This cell requests the public Macrotrends Tesla revenue page and parses the HTML table. If the website changes its HTML or blocks requests, use the revenue table from the course lab and retain the required `Date` and `Revenue` columns.

In [ ]:
url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

tables = pd.read_html(response.text)
tesla_revenue = None

for table in tables:
    cols = [str(c).strip().lower() for c in table.columns]
    if any("date" in c for c in cols) and any("revenue" in c for c in cols):
        tesla_revenue = table.copy()
        break

if tesla_revenue is None:
    raise ValueError("Could not find the Tesla revenue table. Check the course lab's provided scraping method.")

tesla_revenue.columns = [str(c).strip() for c in tesla_revenue.columns]
tesla_revenue = tesla_revenue.rename(columns={
    next(c for c in tesla_revenue.columns if "date" in c.lower()): "Date",
    next(c for c in tesla_revenue.columns if "revenue" in c.lower()): "Revenue"
})
tesla_revenue["Revenue"] = (
    tesla_revenue["Revenue"].astype(str)
    .str.replace(r"[$,]", "", regex=True)
    .replace({"": None, "nan": None})
)
tesla_revenue["Revenue"] = pd.to_numeric(tesla_revenue["Revenue"], errors="coerce")
tesla_revenue = tesla_revenue.dropna(subset=["Revenue"]).reset_index(drop=True)
tesla_revenue.tail()

## Question 3 — Extract GameStop stock data using yfinance

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4 — Extract GameStop revenue data using web scraping

In [ ]:
url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

tables = pd.read_html(response.text)
gme_revenue = None

for table in tables:
    cols = [str(c).strip().lower() for c in table.columns]
    if any("date" in c for c in cols) and any("revenue" in c for c in cols):
        gme_revenue = table.copy()
        break

if gme_revenue is None:
    raise ValueError("Could not find the GameStop revenue table. Check the course lab's provided scraping method.")

gme_revenue.columns = [str(c).strip() for c in gme_revenue.columns]
gme_revenue = gme_revenue.rename(columns={
    next(c for c in gme_revenue.columns if "date" in c.lower()): "Date",
    next(c for c in gme_revenue.columns if "revenue" in c.lower()): "Revenue"
})
gme_revenue["Revenue"] = (
    gme_revenue["Revenue"].astype(str)
    .str.replace(r"[$,]", "", regex=True)
    .replace({"": None, "nan": None})
)
gme_revenue["Revenue"] = pd.to_numeric(gme_revenue["Revenue"], errors="coerce")
gme_revenue = gme_revenue.dropna(subset=["Revenue"]).reset_index(drop=True)
gme_revenue.tail()

## Dashboard helper — make_graph

In [ ]:
def make_graph(stock_data, revenue_data, stock, title):
    stock_data = stock_data.copy()
    revenue_data = revenue_data.copy()

    date_col = "Date" if "Date" in stock_data.columns else stock_data.columns[0]
    stock_data[date_col] = pd.to_datetime(stock_data[date_col], errors="coerce")
    revenue_data["Date"] = pd.to_datetime(revenue_data["Date"], errors="coerce")

    if "Close" not in stock_data.columns:
        raise ValueError("Stock dataframe must contain a 'Close' column.")
    if "Revenue" not in revenue_data.columns:
        raise ValueError("Revenue dataframe must contain a 'Revenue' column.")

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=False,
        subplot_titles=(f"{stock} Historical Share Price", f"{stock} Quarterly Revenue"),
        vertical_spacing=0.18
    )

    fig.add_trace(
        go.Scatter(x=stock_data[date_col], y=stock_data["Close"], name="Share Price"),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(x=revenue_data["Date"], y=revenue_data["Revenue"], name="Revenue"),
        row=2, col=1
    )

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue", row=2, col=1)
    fig.update_layout(title=title, height=800, showlegend=True)
    fig.show()

## Question 5 — Tesla stock and revenue dashboard

In [ ]:
make_graph(tesla_data, tesla_revenue, "TSLA", "Tesla Stock Price and Revenue")

## Question 6 — GameStop stock and revenue dashboard

In [ ]:
make_graph(gme_data, gme_revenue, "GME", "GameStop Stock Price and Revenue")

## Before submitting

- Run all cells in order in an internet-enabled Jupyter environment.
- Confirm that Questions 1 and 3 display the first five stock rows.
- Confirm that Questions 2 and 4 display the last five revenue rows.
- Confirm that both dashboards render.
- Take screenshots of the code and output for Coursera Questions 1–6.
- Save the executed notebook and upload it to your own GitHub repository for Question 7.